# Python Functions: Comprehensive Guide & Reference

Functions are reusable, self-contained blocks of code designed to perform a specific task. In Python, functions are **first-class citizens**, meaning they can be passed as arguments, assigned to variables, and returned from other functions.

In this notebook, we cover:
1. **Function Definition & Anatomical Components**
2. **Types of Functions: With vs Without Parameters & Return Values**
3. **Function Parameters & Arguments**:
   * Positional Arguments
   * Keyword Arguments
   * Default Parameter Values (and the Mutable Default Argument Trap!)
   * Variable-Length Arguments: `*args` (positional) and `**kwargs` (keyword)
   * Positional-Only (`/`) and Keyword-Only (`*`) Parameters (PEP 570 & PEP 3102)
4. **Argument Passing Semantics: Pass-by-Object-Reference / Pass-by-Assignment**
5. **Variable Scope & Namespaces (LEGB Rule, `global`, `nonlocal`)**
6. **First-Class Functions, Closures & Higher-Order Functions**
7. **Anonymous (Lambda) Functions & Functional Tools (`map`, `filter`, `reduce`)**
8. **Recursive Functions & Recursion Limits**
9. **Docstrings & Function Annotations (Type Hints - PEP 484)**
10. **Decorators & Wrappers (`functools.wraps`)**

### 1. Anatomy of a Python Function

**Explanation**: Functions are defined using the `def` keyword, followed by the function name, parentheses for parameters, a colon, an optional docstring, and the indented body.

**Syntax**:
```python
def function_name(param1, param2):
    """Docstring explaining purpose."""
    # body
    return result
```

In [ ]:
# Basic function definition
def greet(name: str) -> str:
    """Return a friendly greeting message."""
    return f"Hello, {name}!"

# Calling the function
msg = greet("World")
print(msg)
print("Docstring:", greet.__doc__)

### 2. Functions: With/Without Parameters & Returns

**Explanation**: Functions can operate with or without input arguments, and with or without an explicit return statement (functions without return implicitly return None).

**Syntax**:
```python
# 1. No params, no return
# 2. With params, no return
# 3. No params, with return
# 4. With params, with return
```

In [ ]:
# 1. No parameters, no return value
def log_startup():
    print("[SYSTEM] System initialized successfully.")

res1 = log_startup()
print("Return value of log_startup():", res1)  # None

# 2. Parameters with multiple return values (returned as a tuple)
def min_max(numbers: list[int]):
    return min(numbers), max(numbers)  # Tuple packing

lowest, highest = min_max([45, 12, 89, 3, 77])
print(f"Lowest: {lowest}, Highest: {highest}")

### 3. Positional vs Keyword Arguments & Defaults

**Explanation**: Positional arguments are matched in order. Keyword arguments are matched by name. Default values allow parameters to be optional.

**Syntax**:
```python
def describe_pet(animal_type, pet_name='Buddy'):
    ...
```

In [ ]:
# Positional & Keyword Arguments
def describe_pet(animal_type: str, pet_name: str = "Buddy"):
    print(f"I have a {animal_type} named {pet_name}.")

# Positional
describe_pet("dog", "Rex")

# Using default
describe_pet("dog")

# Keyword arguments (order does not matter)
describe_pet(pet_name="Whiskers", animal_type="cat")

### 4. The Mutable Default Argument Trap (Crucial!)

**Explanation**: Default parameter expressions are evaluated ONLY ONCE at function definition time, NOT every time the function is called! Using mutable defaults like [] or {} causes shared state across calls.

**Syntax**:
```python
# WRONG: def append_item(item, lst=[]):
# RIGHT: def append_item(item, lst=None):
    if lst is None: lst = []
```

In [ ]:
# The Mutable Default Argument Bug:
def buggy_append(item, target_list=[]):
    target_list.append(item)
    return target_list

print("Call 1:", buggy_append("A"))  # ['A']
print("Call 2:", buggy_append("B"))  # ['A', 'B'] -> Unexpectedly retained!
print("Call 3:", buggy_append("C"))  # ['A', 'B', 'C']

# The Safe, Idiomatic Solution:
def safe_append(item, target_list=None):
    if target_list is None:
        target_list = []
    target_list.append(item)
    return target_list

print("\nSafe Call 1:", safe_append("A"))
print("Safe Call 2:", safe_append("B"))  # ['B'] (Clean fresh list!)

### 5. Variable-Length Arguments (`*args` and `**kwargs`)

**Explanation**: `*args` collects arbitrary extra positional arguments into a tuple. `**kwargs` collects arbitrary keyword arguments into a dictionary.

**Syntax**:
```python
def func(*args, **kwargs):
    # args is a tuple
    # kwargs is a dict
```

In [ ]:
# *args and **kwargs example
def calculate_summary(title, *scores, **metadata):
    print(f"=== {title} ===")
    print("Scores received (tuple) :", scores)
    print("Average score           :", sum(scores) / len(scores) if scores else 0)
    print("Metadata (dict)         :", metadata)

calculate_summary("Math Exam", 85, 90, 78, 92, semester="Spring", year=2026)

### 6. Positional-Only (`/`) & Keyword-Only (`*`) Parameters

**Explanation**: Use `/` to enforce positional-only arguments (before `/`), and `*` to enforce keyword-only arguments (after `*`). Introduced in Python 3.8 (PEP 570) and Python 3.0 (PEP 3102).

**Syntax**:
```python
def func(pos_only, /, standard, *, kw_only):
    ...
```

In [ ]:
# Positional-only & Keyword-only example
def configure_server(host, port, /, timeout=30, *, secure=True, debug=False):
    print(f"Connecting to {host}:{port} (timeout={timeout}s, secure={secure}, debug={debug})")

# host and port MUST be positional
# timeout can be positional or keyword
# secure and debug MUST be specified by keyword!
configure_server("127.0.0.1", 8080, secure=True, debug=True)

try:
    # Passing host as keyword raises TypeError
    configure_server(host="127.0.0.1", port=8080)
except TypeError as e:
    print("Caught expected positional-only error:", e)

### 7. Argument Passing: Pass-by-Object-Reference

**Explanation**: Python is neither pure 'pass-by-value' nor 'pass-by-reference'. It is 'pass-by-assignment': parameters receive references to objects. Mutating mutable objects in place affects caller; reassigning variable rebinds local name.

**Syntax**:
```python
# Rebinding parameter vs in-place mutation
```

In [ ]:
# Pass-by-assignment demonstration
def modify_values(num, lst):
    num = num + 10       # Rebinding local variable: caller's integer is unchanged
    lst.append(999)      # In-place mutation: caller's list is modified!

my_num = 50
my_list = [1, 2, 3]

modify_values(my_num, my_list)

print("Original number (unchanged):", my_num)
print("Original list   (mutated)  :", my_list)

### 8. Variable Scope & The LEGB Rule (`global`, `nonlocal`)

**Explanation**: Python resolves variable names using LEGB: Local -> Enclosing -> Global -> Built-in. Use `global` to modify module-level variables; use `nonlocal` to modify enclosing (outer function) variables.

**Syntax**:
```python
# LEGB resolution
global x
nonlocal y
```

In [ ]:
# Scope & nonlocal example
counter = 0  # Global

def outer():
    count = 10  # Enclosing

    def inner():
        nonlocal count  # Targets enclosing count
        count += 5
        print("Inner count:", count)

    inner()
    print("Outer count after inner():", count)

outer()

### 9. Anonymous (Lambda) Functions & Functional Tools

**Explanation**: Lambdas are small, anonymous functions defined in a single expression. Commonly used with `sorted()`, `map()`, and `filter()`.

**Syntax**:
```python
lambda param1, param2: expression
```

In [ ]:
# Lambda, map, and filter examples
# 1. Lambda for custom sorting
pairs = [(1, "one"), (3, "three"), (2, "two")]
pairs.sort(key=lambda p: p[1])
print("Sorted by second element:", pairs)

# 2. map(): transform elements
nums = [1, 2, 3, 4, 5]
doubled = list(map(lambda x: x * 2, nums))
print("Doubled with map        :", doubled)

# 3. filter(): filter elements by predicate
evens = list(filter(lambda x: x % 2 == 0, nums))
print("Evens with filter       :", evens)

### 10. Recursive Functions & Base Cases

**Explanation**: A function that calls itself to solve smaller instances of the same problem. Requires a **base case** to prevent infinite recursion and `RecursionError`.

**Syntax**:
```python
def factorial(n):
    if n <= 1: return 1  # Base case
    return n * factorial(n - 1)  # Recursive step
```

In [ ]:
# Recursive factorial & Fibonacci
def factorial(n: int) -> int:
    if n <= 1:
        return 1
    return n * factorial(n - 1)

print(f"5! = {factorial(5)}")

# Recursion limit check
import sys
print(f"Python recursion limit: {sys.getrecursionlimit()} frames")

### 11. Function Decorators (`@decorator`)

**Explanation**: A decorator is a higher-order function that takes a function as input, extends or alters its behavior, and returns a new function without modifying the original source code.

**Syntax**:
```python
@decorator
def my_func():
    ...
```

In [ ]:
# Custom execution timing decorator
import time
from functools import wraps

def time_it(func):
    @wraps(func)  # Preserves function name and docstring
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        duration = time.perf_counter() - start_time
        print(f"[TIME] {func.__name__}() took {duration:.6f}s to execute.")
        return result
    return wrapper

@time_it
def compute_squares(n):
    """Compute sum of squares up to n."""
    return sum(i ** 2 for i in range(n))

res = compute_squares(1_000_000)
print("Result:", res)
print("Function name preserved:", compute_squares.__name__)